# NB19 The functions of money and the "zero intrinsic value" equilibrium — Chapter 19

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB19-1: Garratt–Wallace question (iv) — a collapse that comes with probability 1−π each period. The probability of "some time" and the probability of "the very next period" differ (for study)
pi = 0.99                                            # probability of surviving one more period (the paper's example)
print(f"π = {pi}: conditional probability of a collapse in the next period = {1 - pi:.2f}")
print(f"{'n':>6} {'P(collapse within n) 1−π^n':>26} {'P(survive through n) π^n':>26}")
for n in [1, 10, 30, 68, 100, 300, 1000]:
    print(f"{n:>6} {1 - pi**n:>26.3f} {pi**n:>26.3f}")
print("\nThe probability of a collapse within 68 periods is almost one-half, but the probability of a collapse at date 68, seen from date 67, is still 0.01 (the arithmetic on pp. 1891–1892 of the paper).")
print("The date of the collapse follows a geometric distribution, and its expected value is", round(1 / (1 - pi)), "periods: there is a mean but no date.")

π = 0.99: conditional probability of a collapse in the next period = 0.01
     n P(collapse within n) 1−π^n   P(survive through n) π^n
     1                      0.010                      0.990
    10                      0.096                      0.904
    30                      0.260                      0.740
    68                      0.495                      0.505
   100                      0.634                      0.366
   300                      0.951                      0.049
  1000                      1.000                      0.000

The probability of a collapse within 68 periods is almost one-half, but the probability of a collapse at date 68, seen from date 67, is still 0.01 (the arithmetic on pp. 1891–1892 of the paper).
The date of the collapse follows a geometric distribution, and its expected value is 100 periods: there is a mean but no date.


In [ ]:
# NB19-2: Schilling–Uhlig equation (8) — "pick" an initial value Q0 and a mean-zero shock ε, and one equilibrium path is built. There are as many equilibria as there are ways to pick (for study)
# Note: Q is a model variable (unitless). It is not the price of any asset, and the paths below are not forecasts.
import random
random.seed(19)

def construct_path(Q0, steps, eps_draw, cov_term=0.0):
    """Eq. (8): Q_{t+1} = Q_t + ε_{t+1} − cov_t(ε_{t+1}, m_{t+1}) / E_t[m_{t+1}].
    With cov_term=0 this is a martingale (Proposition 1, eq. (2)). With a correlation, cov_term adds an expected change (Corollary 1)."""
    Q = [Q0]
    for _ in range(steps):
        Q.append(Q[-1] + eps_draw() - cov_term)
    return Q

Q0, steps = 100.0, 12
draws = {
    "ε≡0 (Kareken–Wallace constant price)": lambda: 0.0,
    "ε ~ uniform(−5, 5)":                   lambda: random.uniform(-5, 5),
    "ε ∈ {−20, +20}, each 1/2":             lambda: random.choice([-20.0, 20.0]),
}
print(f"Q0 = {Q0}, {steps} periods. Three equilibrium paths from the same Q0 (martingale, cov term 0):")
for name, d in draws.items():
    path = construct_path(Q0, steps, d)
    print(f"  {name:<36} → " + " ".join(f"{q:6.1f}" for q in path))

# Numerical check of the martingale property: the mean over many paths stays near Q0 (it neither rises nor falls in expectation)
N = 20000
end_values = [construct_path(Q0, steps, draws["ε ~ uniform(−5, 5)"])[-1] for _ in range(N)]
print(f"\nMean at the end of period {steps} over {N:,} uniform-shock paths: {sum(end_values)/N:.2f} (Q0 = {Q0}). The mean stays put; only the spread grows")
print(f"  End of period {steps}: min {min(end_values):.1f}, max {max(end_values):.1f}")

# Corollary 1: positive correlation with the pricing kernel gives a supermartingale (falls in expectation), negative a submartingale (rises). The model does not fix the sign.
for cov_term, label in [(1.0, "cov > 0 → supermartingale"), (-1.0, "cov < 0 → submartingale")]:
    mean_end = sum(construct_path(Q0, steps, draws["ε ~ uniform(−5, 5)"], cov_term)[-1] for _ in range(N)) / N
    print(f"  {label}: mean at the end of period {steps} = {mean_end:.2f}")

Q0 = 100.0, 12 periods. Three equilibrium paths from the same Q0 (martingale, cov term 0):
  ε≡0 (Kareken–Wallace constant price) →  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
  ε ~ uniform(−5, 5)                   →  100.0  101.8  104.6  104.8  104.9  103.9  108.8  106.7  103.2  100.8   98.4   96.7   94.4
  ε ∈ {−20, +20}, each 1/2             →  100.0   80.0  100.0  120.0  100.0   80.0   60.0   40.0   20.0   40.0   60.0   40.0   20.0

Mean at the end of period 12 over 20,000 uniform-shock paths: 99.94 (Q0 = 100.0). The mean stays put; only the spread grows
  End of period 12: min 64.7, max 138.4
  cov > 0 → supermartingale: mean at the end of period 12 = 87.89
  cov < 0 → submartingale: mean at the end of period 12 = 112.12
